# Topic 3C: Random Forest for Fraud Detection
**Module 1 - Introduction to Machine Learning in Python**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, f1_score, roc_curve)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


## 1. Create Imbalanced Fraud Dataset


In [ ]:
np.random.seed(42)
n = 50000
fraud_rate = 0.02  # 2% fraud

df = pd.DataFrame({
    'amount': np.random.lognormal(3, 1.5, n).round(2),
    'hour': np.random.randint(0, 24, n),
    'day_of_week': np.random.randint(0, 7, n),
    'merchant_risk': np.random.uniform(0, 1, n),
    'distance_from_home': np.random.exponential(20, n),
    'num_txn_last_hour': np.random.poisson(2, n),
    'avg_amount_ratio': np.random.lognormal(0, 0.5, n),
})

# Fraud more likely for: high amount, unusual hour, high merchant risk
fraud_logit = (-5 + 0.005*df['amount'] + 0.1*(df['hour']<6).astype(int)*3
              + 2*df['merchant_risk'] + 0.02*df['distance_from_home']
              + 0.3*df['num_txn_last_hour'] + np.random.normal(0, 0.5, n))
from scipy.special import expit
df['fraud'] = (np.random.random(n) < expit(fraud_logit)).astype(int)

print(f'Fraud rate: {df["fraud"].mean():.4f} ({df["fraud"].sum()} fraudulent out of {n})')


## 2. Train Random Forest (Without Balancing)


In [ ]:
features = [c for c in df.columns if c != 'fraud']
X = df[features]
y = df['fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Unbalanced model
rf_unbal = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_unbal.fit(X_train, y_train)
proba_unbal = rf_unbal.predict_proba(X_test)[:, 1]

print('=== Without Class Balancing ===')
print(f'AUROC: {roc_auc_score(y_test, proba_unbal):.4f}')
print(f'AUPRC: {average_precision_score(y_test, proba_unbal):.4f}')


## 3. Train with Class Balancing


In [ ]:
# Balanced model
rf_bal = RandomForestClassifier(n_estimators=200, max_depth=10, 
                                class_weight='balanced', random_state=42, n_jobs=-1)
rf_bal.fit(X_train, y_train)
proba_bal = rf_bal.predict_proba(X_test)[:, 1]

print('=== With class_weight=balanced ===')
print(f'AUROC: {roc_auc_score(y_test, proba_bal):.4f}')
print(f'AUPRC: {average_precision_score(y_test, proba_bal):.4f}')


## 4. Train with SMOTE


In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(sampling_strategy=0.3, random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    print(f'Original: {y_train.value_counts().to_dict()}')
    print(f'After SMOTE: {pd.Series(y_resampled).value_counts().to_dict()}')
    
    rf_smote = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    rf_smote.fit(X_resampled, y_resampled)
    proba_smote = rf_smote.predict_proba(X_test)[:, 1]
    print(f'\nAUROC: {roc_auc_score(y_test, proba_smote):.4f}')
    print(f'AUPRC: {average_precision_score(y_test, proba_smote):.4f}')
except ImportError:
    print('Install imbalanced-learn: pip install imbalanced-learn')
    proba_smote = proba_bal  # Fallback


## 5. Model Comparison: ROC and PR Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, proba, color in [('Unbalanced', proba_unbal, 'gray'),
                            ('Balanced', proba_bal, 'steelblue'),
                            ('SMOTE', proba_smote, 'indianred')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auroc = roc_auc_score(y_test, proba)
    ax1.plot(fpr, tpr, label=f'{name} (AUC={auroc:.4f})', color=color, linewidth=2)
    
    prec, rec, _ = precision_recall_curve(y_test, proba)
    auprc = average_precision_score(y_test, proba)
    ax2.plot(rec, prec, label=f'{name} (AUPRC={auprc:.4f})', color=color, linewidth=2)

ax1.plot([0,1],[0,1],'k--', alpha=0.3)
ax1.set_title('ROC Curves'); ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR'); ax1.legend()
ax2.set_title('Precision-Recall Curves'); ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision'); ax2.legend()
plt.tight_layout()
plt.show()


## 6. Feature Importance


In [ ]:
importances = rf_bal.feature_importances_
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color='steelblue')
plt.xlabel('Feature Importance (Gini)')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()


## 7. Threshold Optimization


In [ ]:
prec, rec, thresholds = precision_recall_curve(y_test, proba_bal)
f1_scores = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = f1_scores.max()

print(f'Optimal threshold for F1: {best_threshold:.4f}')
print(f'Best F1 score: {best_f1:.4f}')
print(f'Precision at optimal: {prec[:-1][np.argmax(f1_scores)]:.4f}')
print(f'Recall at optimal: {rec[:-1][np.argmax(f1_scores)]:.4f}')

# Cost-based threshold
cost_fn = 500  # Cost of missed fraud
cost_fp = 10   # Cost of false alarm
total_costs = cost_fn * (1 - rec[:-1]) * y_test.sum() + cost_fp * (1 - prec[:-1]) * (1 - y_test).sum()
cost_threshold = thresholds[np.argmin(total_costs)]
print(f'\nCost-optimal threshold (FN=${cost_fn}, FP=${cost_fp}): {cost_threshold:.4f}')
